# Sequence alignment as a wavefront of tensor updates

*bioinformatics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/bioinformatics/sequence_alignment)

Needleman–Wunsch (global) and Smith–Waterman (local) alignment of DNA sequences by dynamic programming. Every cell of the table needs only its left, upper and diagonal neighbours, so a whole anti-diagonal $i + j = d$ is filled in one tensor update, for a batch of sequence pairs at once. Watch the wavefront sweep the table, then the traceback read off the optimal alignment.

**Vector form**

$$H(i,j) = \max\begin{cases} H(i-1,j-1) + s(a_i, c_j) \\ E(i,j) = \max\{H(i,j-1) - g_o,\ E(i,j-1) - g_e\} \\ F(i,j) = \max\{H(i-1,j) - g_o,\ F(i-1,j) - g_e\} \\ 0 \quad\text{(local only)} \end{cases}$$

**Index (tensor) form**

$$S_{bij} = a_{bi\alpha}\,W_{\alpha\beta}\,c_{bj\beta},\qquad H_{b,\,i_k,\,d-i_k} = \max\big(H_{b,\,i_k-1,\,d-i_k-1} + S_{b,\,i_k,\,d-i_k},\ E_{b,\,i_k,\,d-i_k},\ F_{b,\,i_k,\,d-i_k}\big)\quad \forall\,k \text{ at once}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $a_{bi\alpha},\ c_{bj\beta}$ | `one_hot(codes)` | (B, n, 4), (B, m, 4) | letter $i$ of the first and $j$ of the second sequence of pair $b$, one-hot over A, C, G, T |
| $W_{\alpha\beta}$ | `substitution_matrix(match, mismatch, device)` | (4, 4) | score for aligning letter $\alpha$ with $\beta$: match on the diagonal, mismatch elsewhere |
| $S_{bij} = a_{bi\alpha} W_{\alpha\beta} c_{bj\beta}$ | `einsum('bia,ay,bjy->bij', a, W, c)` | (B, n, m) | repeated $\alpha, \beta$ ⇒ summed: the score of every possible pairing of letters |
| $g_o,\ g_e$ | `gap_open, gap_extend` | scalars | a gap of $k$ letters costs $g_o + (k-1)\,g_e$ (affine); $g_o = g_e$ is a linear gap |
| $(i_k,\ d - i_k)$ | `i = arange(...); j = d - i` | (K,) | all cells on anti-diagonal $d$; they depend only on diagonals $d-1$ and $d-2$ |
| $H, E, F$ | `wavefront_step(H, E, F, S, i, j, ...)` | (B, K) each | one tensor update per diagonal: best score ending in a match, a gap in $a$, a gap in $c$ |
| $H_{b,n_b,m_b},\ \max_{ij} H_{bij}$ | `best_cells(H, len_a, len_c, local)` | (B,), (B, 2) | global score at the corner, local score at the best cell; padding beyond each length is ignored |

Global alignment forces both sequences to be aligned end to end; local alignment adds the option 0, so a bad stretch is simply dropped and the best-scoring piece anywhere is reported (here a motif planted in random background). The traceback walks back from the end cell, at each step asking which of the three options produced the stored value, and switches between the tables $H$, $E$, $F$ while inside a gap. The cost is still $O(nm)$ cells, but only $n + m - 1$ sequential steps: this anti-diagonal parallelism is how GPU and SIMD aligners work. Try raising the mutation rate, or $g_o > g_e$ to favour a few long gaps over many short ones. Proteins use a $20\times20$ matrix such as BLOSUM62 in place of $W$; nothing else changes.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/bioinformatics/sequence_alignment/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import substitution_scores


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    local = params["mode"] == "local"
    go, ge = params["gap_open"], params["gap_extend"]  # a gap of k letters costs go + (k − 1)·ge
    W = tensors.substitution_matrix(params["match"], params["mismatch"], dev)  # (4, 4)
    make = tensors.planted_pairs if local else tensors.related_pairs
    data = make(params["n_pairs"], params["length"], params["mutation"], 0, dev)
    codes_a, len_a, codes_c, len_c = data[:4]  # (B, n), (B,), (B, m), (B,)
    B, n, m = codes_a.shape[0], codes_a.shape[1], codes_c.shape[1]
    print(f"{'Smith–Waterman (local)' if local else 'Needleman–Wunsch (global)'}: {B} pairs, up to {n} × {m} letters")
    print(f"scores: match {params['match']:+g}, mismatch {params['mismatch']:+g}, gap {go:g} + {ge:g} per extra letter")

    S = torch.zeros(B, n + 1, m + 1, device=dev)  # S_{bij}, row and column 0 unused
    S[:, 1:, 1:] = substitution_scores(tensors.one_hot(codes_a), W, tensors.one_hot(codes_c))
    H, E, F = solve.fill(S, go, ge, local)
    print(f"{n + m - 1} wavefront updates fill {B * n * m} cells (one per anti-diagonal instead of one per cell)")

    scores, ends = solve.best_cells(H, len_a, len_c, local)  # (B,), (B, 2)
    print(f"best scores: {[round(s) for s in scores.tolist()]}")
    a, c = tensors.decode(codes_a[0]), tensors.decode(codes_c[0])
    path, top, bottom = solve.traceback(H[0], E[0], F[0], S[0], a, c, tuple(ends[0].tolist()), go, ge, local)
    same = sum(x == y for x, y in zip(top, bottom))
    print(f"pair 0: score {scores[0]:.0f}, alignment length {len(top)}, identity {same}/{len(top)}")
    for k in range(0, len(top), 60):
        print(f"  {top[k : k + 60]}\n  {bottom[k : k + 60]}")
    if local:
        span = data[4][0].tolist()
        (i0, j0), (i1, j1) = path[0], path[-1]
        print(f"found a[{i0}:{i1}] ~ c[{j0}:{j1}]; motif planted at a[{span[0]}:{span[1]}], c[{span[2]}:{span[3]}]")

    done_at = len_a + len_c  # anti-diagonal on which each pair's table is complete
    return plot.animate(
        H[0, : len_a[0] + 1, : len_c[0] + 1], path, a, c, top, bottom, scores, done_at, local, out_dir / "alignment.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/bioinformatics/sequence_alignment/equation.py
r"""Alignment rules → tensor translation.

Two DNA sequences a (length n) and c (length m) are aligned by dynamic programming. With one-hot residues
a_{bi\alpha} (pair b, position i, letter \alpha) and the substitution matrix W_{\alpha\beta}, every match score is

    S_{bij} = a_{bi\alpha}\,W_{\alpha\beta}\,c_{bj\beta}        (\alpha, \beta summed over A, C, G, T)

A gap of length k costs g_o + (k − 1) g_e (affine; g_o = g_e is a linear gap). Gotoh's recurrences keep three tables:
H (best score ending at cell i, j), E (ending with a gap in a) and F (ending with a gap in c):

    E_{bij} = max(H_{b,i,j-1} − g_o,  E_{b,i,j-1} − g_e)
    F_{bij} = max(H_{b,i-1,j} − g_o,  F_{b,i-1,j} − g_e)
    H_{bij} = max(H_{b,i-1,j-1} + S_{bij},  E_{bij},  F_{bij}  [, 0 for Smith–Waterman])

Cell (i, j) only needs cells with a smaller i + j. So every cell on one anti-diagonal i + j = d can be computed at
the same time from the two previous diagonals: the "wavefront". One diagonal is one tensor update, for every cell
on it and every pair b in the batch.
"""

import torch


def substitution_scores(a: torch.Tensor, w: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    r"""S_{bij} = a_{bi\alpha} W_{\alpha\beta} c_{bj\beta}   (\alpha, \beta summed): (B, n, 4), (B, m, 4) → (B, n, m)"""
    return torch.einsum("bia,ay,bjy->bij", a, w, c)


def wavefront_step(H, E, F, S, i, j, gap_open: float, gap_extend: float, local: bool):
    r"""New H, E, F on one anti-diagonal: cells (i_k, j_k) with i_k + j_k = d, for every pair at once.

    H, E, F, S: (B, n+1, m+1) tables (row and column 0 are the empty prefix); i, j: (K,) cell indices.
    Returns three (B, K) tensors.
    """
    e = torch.maximum(H[:, i, j - 1] - gap_open, E[:, i, j - 1] - gap_extend)  # gap in a: step from the left
    f = torch.maximum(H[:, i - 1, j] - gap_open, F[:, i - 1, j] - gap_extend)  # gap in c: step from above
    h = torch.maximum(torch.maximum(H[:, i - 1, j - 1] + S[:, i, j], e), f)  # match/mismatch: step diagonally
    if local:
        h = h.clamp(min=0.0)  # Smith–Waterman: a local alignment may start afresh anywhere
    return h, e, f


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/bioinformatics/sequence_alignment/tensors.py
"""Build the concrete tensors: the DNA substitution matrix and a batch of related sequence pairs (synthetic, seeded)."""

import torch
import torch.nn.functional as F

ALPHABET = "ACGT"
PAD = 4  # code of the padding symbol after the end of a shorter sequence


def substitution_matrix(match: float, mismatch: float, device: torch.device) -> torch.Tensor:
    """W_{αβ}: `match` on the diagonal, `mismatch` everywhere else: (4, 4)."""
    eye = torch.eye(4, device=device)
    return match * eye + mismatch * (1 - eye)


def encode(seq: str) -> torch.Tensor:
    """'GATTACA' → tensor([2, 0, 3, 3, 0, 1, 0])."""
    return torch.tensor([ALPHABET.index(ch) for ch in seq])


def decode(codes: torch.Tensor) -> str:
    return "".join(ALPHABET[k] for k in codes.tolist() if k < 4)


def pad(seqs: list[torch.Tensor], device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Stack sequences of different lengths: codes (B, L_max) padded with PAD, and lengths (B,)."""
    lengths = torch.tensor([len(s) for s in seqs])
    codes = torch.full((len(seqs), int(lengths.max())), PAD, dtype=torch.long)
    for b, s in enumerate(seqs):
        codes[b, : len(s)] = s
    return codes.to(device), lengths.to(device)


def one_hot(codes: torch.Tensor) -> torch.Tensor:
    """a_{biα}: (B, L) codes → (B, L, 4); padding becomes an all-zero row (it scores 0 and is never used)."""
    return F.one_hot(codes, 5)[..., :4].float()


def mutate(seq: torch.Tensor, rate: float, gen: torch.Generator) -> torch.Tensor:
    """Substitute a fraction `rate` of the letters and delete a fraction rate/4 of the sites."""
    flip = torch.rand(len(seq), generator=gen) < rate
    new = torch.randint(0, 4, (len(seq),), generator=gen)
    out = torch.where(flip, new, seq)
    keep = torch.rand(len(seq), generator=gen) >= rate / 4
    return out[keep]


def related_pairs(n_pairs: int, length: int, rate: float, seed: int, device: torch.device):
    """Global case: both sequences of a pair descend from one random ancestor (substitutions + indels).

    A deletion in one descendant is an insertion as seen from the other. Returns codes_a, len_a, codes_c, len_c.
    """
    gen = torch.Generator().manual_seed(seed)
    a, c = [], []
    for _ in range(n_pairs):
        ancestor = torch.randint(0, 4, (length,), generator=gen)
        a.append(mutate(ancestor, rate, gen))
        c.append(mutate(ancestor, rate, gen))
    return (*pad(a, device), *pad(c, device))


def planted_pairs(n_pairs: int, length: int, rate: float, seed: int, device: torch.device):
    """Local case: unrelated random sequences that share one planted motif (length/2) at random positions.

    Returns codes_a, len_a, codes_c, len_c and the planted spans (B, 4) = (start_a, end_a, start_c, end_c).
    """
    gen = torch.Generator().manual_seed(seed)
    a, c, spans = [], [], []
    k = max(6, length // 2)
    for _ in range(n_pairs):
        motif = torch.randint(0, 4, (k,), generator=gen)
        row = []
        for out in (a, c):
            copy = mutate(motif, rate, gen)
            start = int(torch.randint(0, length - k + 1, (1,), generator=gen))
            background = torch.randint(0, 4, (length - k,), generator=gen)
            out.append(torch.cat([background[:start], copy, background[start:]]))
            row += [start, start + len(copy)]
        spans.append(row)
    return (*pad(a, device), *pad(c, device), torch.tensor(spans))


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/bioinformatics/sequence_alignment/solve.py
"""Fill the DP tables diagonal by diagonal for the whole batch, read off the scores and trace back the alignment."""

import torch

from problems._common.progress import steps

from .equation import wavefront_step


def init_tables(S: torch.Tensor, gap_open: float, gap_extend: float, local: bool):
    """H, E, F of shape (B, n+1, m+1) with the empty-prefix row and column filled in."""
    B, n1, m1 = S.shape
    H = torch.zeros(B, n1, m1, device=S.device)
    E = torch.full_like(H, float("-inf"))
    F = torch.full_like(H, float("-inf"))
    if not local:  # Needleman–Wunsch: a prefix aligned to nothing is one gap of that length
        gap = lambda k: -(gap_open + (k - 1) * gap_extend)  # noqa: E731
        H[:, 0, 1:] = gap(torch.arange(1, m1, device=S.device))
        H[:, 1:, 0] = gap(torch.arange(1, n1, device=S.device))
        E[:, 0, 1:] = H[:, 0, 1:]
        F[:, 1:, 0] = H[:, 1:, 0]
    return H, E, F


def fill(S: torch.Tensor, gap_open: float, gap_extend: float, local: bool):
    """Run the wavefront over the anti-diagonals d = 2 … n+m. S: (B, n+1, m+1) → H, E, F, each (B, n+1, m+1)."""
    H, E, F = init_tables(S, gap_open, gap_extend, local)
    n, m = S.shape[1] - 1, S.shape[2] - 1
    for d in steps(n + m - 1, "anti-diagonals"):
        d += 2
        i = torch.arange(max(1, d - m), min(n, d - 1) + 1, device=S.device)  # (K,) rows on this diagonal
        j = d - i  # (K,) matching columns
        H[:, i, j], E[:, i, j], F[:, i, j] = wavefront_step(H, E, F, S, i, j, gap_open, gap_extend, local)
    return H, E, F


def best_cells(H: torch.Tensor, len_a: torch.Tensor, len_c: torch.Tensor, local: bool):
    """Score and end cell of every pair: (B,), (B, 2).

    Global: the bottom-right cell (n_b, m_b) of each pair's own lengths. Local: the best cell of that pair's table.
    """
    B = H.shape[0]
    if not local:
        ends = torch.stack([len_a, len_c], dim=1)
        return H[torch.arange(B), len_a, len_c], ends
    rows = torch.arange(H.shape[1], device=H.device)
    cols = torch.arange(H.shape[2], device=H.device)
    inside = (rows[None, :, None] <= len_a[:, None, None]) & (cols[None, None, :] <= len_c[:, None, None])
    flat = torch.where(inside, H, float("-inf")).flatten(1)  # padding cells are ignored
    score, k = flat.max(dim=1)
    return score, torch.stack([k // H.shape[2], k % H.shape[2]], dim=1)


def traceback(H, E, F, S, a: str, c: str, end: tuple[int, int], gap_open, gap_extend, local: bool):
    """Walk back from the end cell of one pair, re-deriving each choice from the tables.

    Returns the path [(i, j), …] from start to end and the two aligned strings with '-' for gaps.
    """
    H, E, F, S = (t.cpu().tolist() for t in (H, E, F, S))  # one pair: (n+1, m+1) nested lists
    i, j = end
    state, path, top, bottom = "H", [(i, j)], [], []
    while i > 0 or j > 0:
        if local and state == "H" and H[i][j] <= 0:
            break
        if state == "H" and i > 0 and j > 0 and H[i][j] == H[i - 1][j - 1] + S[i][j]:
            top.append(a[i - 1])
            bottom.append(c[j - 1])
            i, j = i - 1, j - 1
        elif state == "H":
            state = "E" if j > 0 and (i == 0 or H[i][j] == E[i][j]) else "F"
            continue
        elif state == "E":  # gap in a: c_j is aligned to '-'
            state = "H" if i == 0 or E[i][j] == H[i][j - 1] - gap_open else "E"
            top.append("-")
            bottom.append(c[j - 1])
            j -= 1
        else:  # gap in c: a_i is aligned to '-'
            state = "H" if j == 0 or F[i][j] == H[i - 1][j] - gap_open else "F"
            top.append(a[i - 1])
            bottom.append("-")
            i -= 1
        path.append((i, j))
    return path[::-1], "".join(top[::-1]), "".join(bottom[::-1])


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/bioinformatics/sequence_alignment/plot.py
"""Animate the DP table of pair 0 filling one anti-diagonal per frame, then the optimal path being traced back."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(H: torch.Tensor, path: list, a: str, c: str, top: str, bottom: str, scores: torch.Tensor,
            done_at: torch.Tensor, local: bool, out: Path) -> Path:  # fmt: skip
    """H (n+1, m+1) table of pair 0; path [(i, j), …]; scores (B,) and the diagonal at which each pair is done."""
    table = H.cpu().numpy()
    n1, m1 = table.shape
    diag = np.add.outer(np.arange(n1), np.arange(m1))  # i + j of every cell
    ends = done_at.cpu().numpy()  # anti-diagonal on which each pair's table is complete
    n_diag = int(ends.max()) + 1  # the batch is done when its largest table is
    n_path = max(2, len(path) // 2)
    total = n_diag + n_path + 6  # fill, trace back, hold

    fig = plt.figure(figsize=(9, 4.6))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.15, 1], height_ratios=[1, 0.9])
    ax = fig.add_subplot(grid[:, 0])
    ax_s = fig.add_subplot(grid[0, 1])
    ax_t = fig.add_subplot(grid[1, 1])

    vmax = max(1.0, float(np.abs(table[np.isfinite(table)]).max()))
    cmap = plt.get_cmap("Blues" if local else "RdBu_r", 12).with_extremes(bad="#f3f4f6")  # unfilled cells: grey
    image = ax.imshow(np.ma.masked_all(table.shape), cmap=cmap, vmin=0 if local else -vmax, vmax=vmax)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.03, label="score $H_{ij}$")
    ax.grid(False)
    ax.set_xlabel("sequence c  →  j")
    ax.set_ylabel("sequence a  →  i")
    (front,) = ax.plot([], [], color="#f59e0b", lw=2)
    (trace,) = ax.plot([], [], color="#dc2626", lw=2)
    title = ax.set_title(" ", fontsize=10)  # placeholder so tight_layout leaves room

    b = np.arange(len(scores))
    bars = ax_s.bar(b, scores.cpu().numpy(), color="#93c5fd", edgecolor="#1e40af")
    ax_s.set_xlabel("pair $b$")
    ax_s.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax_s.set_ylabel("best score")
    ax_s.set_title("one tensor update per diagonal, all pairs at once", fontsize=9)
    ax_s.axhline(0, color="#9ca3af", lw=0.8)
    lo, hi = min(0.0, float(scores.min())), float(scores.max())
    ax_s.set_ylim(lo - 0.1 * (hi - lo + 1), hi + 0.1 * (hi - lo + 1))

    ax_t.axis("off")
    width = 40
    marks = "".join("|" if x == y else ("·" if "-" not in (x, y) else " ") for x, y in zip(top, bottom))
    lines = []
    for k in range(0, len(top), width):
        lines += [top[k : k + width], marks[k : k + width], bottom[k : k + width], ""]
    text = ax_t.text(0, 1, "", va="top", family="monospace", fontsize=7.5, transform=ax_t.transAxes)

    def update(f: int):
        d = min(f, n_diag - 1)
        ax.images[0].set_data(np.ma.masked_where(diag > d, table))
        if f < n_diag:
            ii = np.arange(max(0, d - m1 + 1), min(n1 - 1, d) + 1)
            front.set_data(d - ii, ii)
            title.set_text(f"pair 0, anti-diagonal i + j = {d}  (batch: {d} of {n_diag - 1})")
        else:
            front.set_data([], [])
            k = min(len(path), round((f - n_diag + 1) * len(path) / n_path))
            pts = path[::-1][:k]
            trace.set_data([p[1] for p in pts], [p[0] for p in pts])
            title.set_text(f"pair 0: traceback of the optimal alignment, score {scores[0]:.0f}")
        for bar, e in zip(bars, ends):
            bar.set_alpha(1.0 if d >= e else 0.15)
        text.set_text("\n".join(lines[:16]) if f >= n_diag + n_path else "")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(total, 75), out, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'mode': 'global',
    'length': 40,
    'n_pairs': 16,
    'mutation': 0.2,
    'match': 2.0,
    'mismatch': -2.0,
    'gap_open': 3.0,
    'gap_extend': 1.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.bioinformatics.sequence_alignment")]:
    del sys.modules[name]
main = importlib.import_module("problems.bioinformatics.sequence_alignment.main")
problem_dir = Path("problems/bioinformatics/sequence_alignment")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
